# SAR-Lab · Yutao Pure training
[Open in Colab](https://colab.research.google.com/github/Ricky-S-Gong/search-rec-ads-portfolio/blob/feature/yutao-kuairand-colab/research/kuairand/notebooks/colab.ipynb)
Code is now in the team repository. Select GPU, run preparation cells in order, and manually start training only after ready_for_training=true.
代码已整理到团队仓库；先连接 GPU，通过预检查后手动训练。当前是 Pure baseline，1K 尚未验收。


In [ ]:
# 1. Mount the Drive belonging to this Colab account (authorize manually).
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Clone team code and freeze the selected branch to its exact commit.
import subprocess
from pathlib import Path


def run_command(command, cwd=None, check=True):
    import os
    environment = dict(os.environ, PYTHONUNBUFFERED="1")
    with subprocess.Popen(command, cwd=cwd, env=environment, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if check and returncode:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode

REPO_DIR = Path("/content/sar-team-code")
REPO_URL = "https://github.com/Ricky-S-Gong/search-rec-ads-portfolio.git"
REF = "feature/yutao-kuairand-colab"  # Change to main after the PR is merged.
# For an exact rerun, paste the originally printed 40-character commit here.
COMMIT = ""
if not COMMIT:
    remote = subprocess.check_output(["git", "ls-remote", REPO_URL, "refs/heads/" + REF], text=True)
    if not remote.strip():
        raise ValueError("Selected branch not found; set REF or an exact COMMIT")
    COMMIT = remote.split()[0]
if len(COMMIT) != 40 or any(c not in "0123456789abcdef" for c in COMMIT):
    raise ValueError("A full immutable commit SHA is required")
if REPO_DIR.exists():
    raise ValueError("Checkout already exists; verify it before reuse, or use a fresh runtime")
run_command(["git", "clone", REPO_URL, str(REPO_DIR)])
run_command(["git", "-C", str(REPO_DIR), "checkout", "--detach", COMMIT])
PROJECT_DIR = REPO_DIR / "research/kuairand"
print("Source commit:", COMMIT)
print("Project directory:", PROJECT_DIR)


In [ ]:
# 3. Recreate the locked project environment; notebook kernel is kept separate.
import os
import sys

run_command([sys.executable, "-m", "pip", "install", "uv==0.12.18"], check=True)
os.environ["UV_CACHE_DIR"] = "/content/sar-uv-cache"
os.environ["UV_PYTHON_INSTALL_DIR"] = "/content/sar-uv-python"
run_command(["uv", "sync", "--locked"], cwd=PROJECT_DIR, check=True)
PYTHON = str(PROJECT_DIR / ".venv/bin/python")
CLI = str(REPO_DIR / "research/kuairand/run.py")
CONFIG = str(REPO_DIR / "research/kuairand/configs/pure.yaml")
run_command([PYTHON, CLI, "check-env"], cwd=PROJECT_DIR, check=True)
run_command([PYTHON, CLI, "check-config", "--config", CONFIG], cwd=PROJECT_DIR, check=True)

初始化和数据检查可以用 CPU；训练前再连接 GPU 并确认 CUDA。托管 kernel 负责 Drive 鉴权，锁定 venv 负责数据适配和训练。所有单元手动执行。


In [ ]:
# W&B key: needed only for the optional connection check and real training.
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
RUN_DIR = "/content/drive/MyDrive/SAR/runs/yutao/w1-pure-integrated-v2-001"
RESUME = ""


## 手动准备 Ziqi 的 Pure 数据

下载使用个人账号现有共享权限。首次可能要求 Google 授权；不会启动训练，不需 A100。若发布文件发生变化，校验会停止并要求重新核验。不要改校验值来跳过检查。


In [ ]:
import runpy

SOURCE_DIR = Path("/content/ziqi-pure-source")
fetcher = runpy.run_path(str(REPO_DIR / "research/kuairand/colab_fetch_pure.py"))
fetcher["fetch_pure"](SOURCE_DIR, REPO_DIR / "research/kuairand/configs/drive-pure-source.json")


In [ ]:
PREPARED_DIR = Path("/content/yutao-pure-integrated-v2")
# Use a fresh output directory. Each new runtime rebuilds the private cache.
run_command([PYTHON, CLI, "prepare-pure", "--config", CONFIG, "--source", str(SOURCE_DIR), "--output", str(PREPARED_DIR)], cwd=REPO_DIR)
MANIFEST = str(PREPARED_DIR / "manifest.json")
run_command([PYTHON, CLI, "check-data", "--config", CONFIG, "--manifest", MANIFEST], cwd=REPO_DIR)


## GPU 训练前检查

已接入真实物品塔、严格历史和冻结特征映射。须连接 GPU 并通过此单元，确认 ready_for_training=true 后，再手动启动训练。预检查做前向和反向，不更新参数、不创建训练 run。


In [ ]:
run_command([PYTHON, CLI, "preflight", "--config", CONFIG, "--manifest", MANIFEST], cwd=REPO_DIR)


可选连接检查：手动运行下一单元会在 SAR-Lab/kuairand-two-tower 创建 setup run，不训练模型。该 run 与后续训练 run 分开。


In [ ]:
run_command([PYTHON, CLI, "init-wandb", "--config", CONFIG], cwd=REPO_DIR, check=True)

## 手动启动真实训练

运行前确认：manifest 校验通过、物品塔代码已接入、数据时间协议已核验、Drive 输出可写。默认配置使用 CUDA 和在线 W&B。第一次运行用新的 RUN_DIR；恢复时使用原目录和原配置。

启动后 10–15 分钟检查 loss、warning、数据等待、显存和持久 checkpoint。本 Notebook 不自动启动或监控实验。


In [ ]:
run_command([PYTHON, CLI, "preflight", "--config", CONFIG, "--manifest", MANIFEST], cwd=REPO_DIR)
if not MANIFEST or not Path(MANIFEST).is_file():
    raise ValueError("Set the real published MANIFEST path")
command = [PYTHON, CLI, "train", "--config", CONFIG, "--manifest", MANIFEST, "--output", RUN_DIR]
if RESUME:
    command += ["--resume", RESUME]
run_command(command, cwd=REPO_DIR, check=True)

## 导出视频向量交给 Ricky

以下采用半开 ID 区间。使用同一配置、manifest 和完整 checkpoint；导出目录必须是新的。候选表须由 Ziqi 发布为全局按 ID 排序且唯一的 Parquet。Ricky 负责资格、检索及评估。不要把向量导出等同于完成 Stage 1 评估。


In [ ]:
import json

with open(Path(RUN_DIR) / "run_manifest.json") as f:
    CHECKPOINT = json.load(f)["checkpoint"]
print("Checkpoint:", CHECKPOINT)
EXPORT_DIR = "/content/drive/MyDrive/SAR/releases/stage1/yutao-vectors-001"
START_ID = 0
END_ID = 7583  # Verified Pure candidate interval [0,7583).
if not CHECKPOINT or END_ID is None:
    raise ValueError("Set checkpoint and candidate ID interval")
run_command(
    [
        PYTHON,
        CLI,
        "export",
        "--config",
        CONFIG,
        "--manifest",
        MANIFEST,
        "--checkpoint",
        CHECKPOINT,
        "--output",
        EXPORT_DIR,
        "--start-id",
        str(START_ID),
        "--end-id",
        str(END_ID),
    ],
    cwd=REPO_DIR,
    check=True,
)